# Simple FRED-MD + ALFRED Builder

1. Non-revised track: latest FRED-MD full history + direct tcode transforms.
2. Real-time track: ALFRED-based strict and balanced snapshots using 9-observation tails, with explicit source tags and availability metadata.

Special rules in this notebook:
- Global anchor-vintage backfill (all series, including ALFRED): detect the last observation month that is revised for each series, choose the first vintage at or after that month, and fill missing earlier months from that anchor to reduce leakage.
- If anchor-vintage data is unavailable for a series, use a conservative local fallback only for the earliest missing pre-history months.

Outputs:
- CSV files in data/ALFRED/simple_outputs/
- Final workbook data/ALFRED/fred_md_simple_balanced_dataset.xlsx

In [1]:
from __future__ import annotations

import importlib.util
import json
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path('/Users/OlavNikolai/TIO4900-Replication')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))


def clean_text(value) -> str:
    if value is None:
        return ''
    if pd.isna(value):
        return ''
    return str(value).strip()


def latest_fred_md_sheet_name(xlsx_path: Path) -> str:
    xls = pd.ExcelFile(xlsx_path)
    for sheet in xls.sheet_names:
        probe = pd.read_excel(xlsx_path, sheet_name=sheet, nrows=3, header=None)
        if probe.empty:
            continue
        first = clean_text(probe.iloc[0, 0]).lower() if probe.shape[0] >= 1 and probe.shape[1] >= 1 else ''
        second = clean_text(probe.iloc[1, 0]).lower() if probe.shape[0] >= 2 and probe.shape[1] >= 1 else ''
        if first == 'sasdate' and second == 'transform:':
            return sheet
    raise ValueError(f'No latest FRED-MD worksheet with sasdate/Transform rows found in {xlsx_path}')


def load_latest_fred_md_matrix(source_path: Path) -> tuple[pd.DataFrame, list[str], list[int]]:
    suffix = source_path.suffix.lower()
    if suffix == '.csv':
        with source_path.open() as handle:
            header = handle.readline().strip().split(';')
            transforms = handle.readline().strip().split(';')
        if clean_text(header[0]).lower() != 'sasdate':
            raise ValueError(f'Unexpected first column in {source_path}: {header[0]}')
        if clean_text(transforms[0]).lower() != 'transform:':
            raise ValueError(f'Unexpected transform row in {source_path}: {transforms[0]}')
        series_names = [clean_text(x) for x in header[1:]]
        tcodes = [int(float(x)) for x in transforms[1:]]
        panel = pd.read_csv(source_path, sep=';', decimal=',', skiprows=[1])
    elif suffix == '.xlsx':
        sheet = latest_fred_md_sheet_name(source_path)
        raw = pd.read_excel(source_path, sheet_name=sheet, header=0)
        raw = raw.rename(columns={raw.columns[0]: 'sasdate'})
        if clean_text(raw.iloc[0, 0]).lower() != 'transform:':
            raise ValueError(f'Unexpected transform row in {source_path}: {raw.iloc[0, 0]}')
        series_names = [clean_text(c) for c in raw.columns[1:]]
        tcodes = [int(float(raw.iloc[0][c])) for c in raw.columns[1:]]
        panel = raw.iloc[1:].copy()
    else:
        raise ValueError(f'Unsupported latest FRED-MD source: {source_path}')

    panel['sasdate'] = pd.to_datetime(panel['sasdate'], errors='coerce')
    panel = panel.dropna(subset=['sasdate']).copy()
    panel['sasdate'] = (panel['sasdate'] + pd.offsets.MonthEnd(0)).dt.normalize()
    panel = panel.sort_values('sasdate').reset_index(drop=True)
    return panel, series_names, tcodes


legacy_path = ROOT / 'data/ALFRED/helpers/revised_data_pipeline.py'
spec = importlib.util.spec_from_file_location('revised_data_pipeline', legacy_path)

hs = importlib.util.module_from_spec(spec)
sys.modules['revised_data_pipeline'] = hs
spec.loader.exec_module(hs)

LATEST_MD_PATH = ROOT / 'data/ALFRED/2026-02-MD.xlsx'
OUT_DIR = ROOT / 'data/ALFRED/simple_outputs'
CACHE_PATH = ROOT / 'data/ALFRED/fred_md_simple_rt_cache.xlsx'
FINAL_XLSX = ROOT / 'data/ALFRED/fred_md_simple_balanced_dataset.xlsx'

USE_SAVED_CACHE = False
MAX_RELEASE_GAP_MONTHS = 1
API_KEY = os.getenv('FRED_API_KEY', '3c268be920acd9693b77b400b0a95cf2')

OUT_DIR.mkdir(parents=True, exist_ok=True)

## Step 1: Load Latest FRED-MD And Build Non-Revised Baseline

This cell reads the latest local FRED-MD file, extracts the series names and transformation codes, and constructs the non-revised raw and transformed panels.

It also writes two baseline CSV files to `simple_outputs` for traceability: full non-revised raw history and full non-revised transformed history.

In [2]:

panel, series_names, tcodes = load_latest_fred_md_matrix(LATEST_MD_PATH)

nonrev_raw = panel.set_index('sasdate')[series_names].apply(pd.to_numeric, errors='coerce').sort_index()
tcode_map = dict(zip(series_names, [int(x) for x in tcodes]))

def tcode_transform(x: pd.Series, tcode: int) -> pd.Series:
    x = pd.to_numeric(x, errors='coerce')
    if tcode == 1:
        return x
    if tcode == 2:
        return x.diff(1)
    if tcode == 3:
        return x.diff(1).diff(1)
    if tcode == 4:
        return np.log(x.where(x > 0))
    if tcode == 5:
        z = x.where(x > 0)
        return np.log(z).diff(1)
    if tcode == 6:
        z = x.where(x > 0)
        return np.log(z).diff(1).diff(1)
    if tcode == 7:
        return (x / x.shift(1) - 1.0).diff(1)
    raise ValueError(f'Unsupported tcode: {tcode}')

nonrev_tcode = pd.DataFrame({c: tcode_transform(nonrev_raw[c], tcode_map[c]) for c in nonrev_raw.columns}, index=nonrev_raw.index)

nonrev_raw.to_csv(OUT_DIR / 'nonrevised_raw_full_history.csv', index_label='sasdate')
nonrev_tcode.to_csv(OUT_DIR / 'nonrevised_tcode_full_history.csv', index_label='sasdate')
print({'series': len(series_names), 'rows': len(nonrev_raw), 'raw_csv': str(OUT_DIR / 'nonrevised_raw_full_history.csv')})

{'series': 126, 'rows': 805, 'raw_csv': '/Users/OlavNikolai/TIO4900-Replication/data/ALFRED/simple_outputs/nonrevised_raw_full_history.csv'}


## Step 2: Build Mapping Registry

This cell builds the series mapping registry used by the real-time pipeline stage in the helper module.

The registry captures construction rules (direct, alias, formula, local-only) and is exported to CSV for auditing.

In [3]:
# Registry mapping is centralized in    revised_data_pipeline.py and mirrors 2026-02-series.md rules.
registry = hs.build_simple_registry(series_names=series_names, tcode_map=tcode_map)
registry.to_csv(OUT_DIR / 'mapping_registry.csv', index=False)
print({
    'registry_rows': len(registry),
    'construction_type_counts': registry['construction_type'].value_counts().to_dict(),
})

{'registry_rows': 126, 'construction_type_counts': {'direct_alfred': 121, 'derived_formula': 2, 'public_proxy': 2, 'direct_fred': 1}}


## Step 3: Fetch Or Load Real-Time Cache And Build Final Outputs

This is the main pipeline cell. It either loads an existing cache workbook or rebuilds it from ALFRED API calls, then creates balanced real-time panels.

It applies anchor-vintage backfill, computes transformed real-time data, writes diagnostic CSVs, and produces the final workbook with summary sheets.

In [4]:
spec = importlib.util.spec_from_file_location('revised_data_pipeline', legacy_path)
hs = importlib.util.module_from_spec(spec)
sys.modules['revised_data_pipeline'] = hs
spec.loader.exec_module(hs)
print('Reloaded revised_data_pipeline helper from disk.')

Reloaded revised_data_pipeline helper from disk.


In [ ]:
from time import perf_counter, sleep

start = nonrev_raw.index.min().strftime('%Y-%m-%d')
end = nonrev_raw.index.max().strftime('%Y-%m-%d')

API_LOG_EVERY_N = 1
API_MIN_INTERVAL_S = 0.55
API_COOLDOWN_ON_429_S = 12.0
API_MAX_INTERVAL_S = 1.50

api_stats = {'calls': 0, 'ok': 0, 'err': 0, 'total_s': 0.0}
endpoint_counts: dict[str, int] = {}
throttle = {'last_call_ts': 0.0, 'min_interval_s': API_MIN_INTERVAL_S}
_orig_fred_get = hs.fred_get

def _log_api(msg: str) -> None:
    ts = pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')
    print(f'[{ts}] {msg}')

def _paced_call() -> None:
    now = perf_counter()
    wait_needed = throttle['min_interval_s'] - (now - throttle['last_call_ts'])
    if wait_needed > 0:
        sleep(wait_needed)

def _logged_fred_get(ctx, endpoint: str, **params):
    series_id = str(params.get('series_id', ''))
    api_stats['calls'] += 1
    call_no = api_stats['calls']

    _paced_call()
    t0 = perf_counter()
    try:
        out = _orig_fred_get(ctx, endpoint, **params)
        dt = perf_counter() - t0
        throttle['last_call_ts'] = perf_counter()

        api_stats['ok'] += 1
        api_stats['total_s'] += dt
        endpoint_counts[endpoint] = endpoint_counts.get(endpoint, 0) + 1

        if call_no == 1 or call_no % API_LOG_EVERY_N == 0:
            _log_api(
                f"API OK #{call_no} endpoint={endpoint} series={series_id or '-'} "
                f"dt={dt:.2f}s ok={api_stats['ok']} err={api_stats['err']} "
                f"min_interval={throttle['min_interval_s']:.2f}s"
            )
        return out
    except Exception as exc:
        dt = perf_counter() - t0
        throttle['last_call_ts'] = perf_counter()

        api_stats['err'] += 1
        api_stats['total_s'] += dt
        endpoint_counts[endpoint] = endpoint_counts.get(endpoint, 0) + 1

        msg = str(exc)
        if '429' in msg:
            throttle['min_interval_s'] = min(throttle['min_interval_s'] + 0.10, API_MAX_INTERVAL_S)
            _log_api(
                f"RATE LIMIT HIT: raising min_interval to {throttle['min_interval_s']:.2f}s "
                f"and cooling down {API_COOLDOWN_ON_429_S:.0f}s"
            )
            sleep(API_COOLDOWN_ON_429_S)

        _log_api(
            f"API ERROR #{call_no} endpoint={endpoint} series={series_id or '-'} "
            f"dt={dt:.2f}s error={type(exc).__name__}: {exc}"
        )
        raise

if USE_SAVED_CACHE and CACHE_PATH.exists():
    _log_api(f'Using saved cache at {CACHE_PATH}; no ALFRED API calls will be made.')
    cache_payload = hs.load_cache(CACHE_PATH)
    cache_mode = 'loaded'
else:
    if not API_KEY:
        raise RuntimeError('FRED_API_KEY is required when rebuilding cache.')
    hs.fred_get = _logged_fred_get
    stage0 = perf_counter()
    _log_api(
        f"Starting ALFRED API fetch stage (run_cache_stage). "
        f"base_min_interval={API_MIN_INTERVAL_S:.2f}s cooldown_on_429={API_COOLDOWN_ON_429_S:.0f}s"
    )
    try:
        cache_payload = hs.run_cache_stage(
            registry=registry,
            cache_path=CACHE_PATH,
            start=start,
            end=end,
            api_key=API_KEY,
            pause_sec=0.20,
        )
        cache_mode = 'rebuilt'
    except Exception:
        elapsed = perf_counter() - stage0
        avg = (api_stats['total_s'] / api_stats['calls']) if api_stats['calls'] else 0.0
        _log_api(
            f"ALFRED API fetch FAILED after {elapsed:.1f}s "
            f"calls={api_stats['calls']} ok={api_stats['ok']} err={api_stats['err']} avg_api={avg:.2f}s"
        )
        if endpoint_counts:
            _log_api(f'Endpoint counts so far: {endpoint_counts}')
        raise
    finally:
        hs.fred_get = _orig_fred_get

    elapsed = perf_counter() - stage0
    avg = (api_stats['total_s'] / api_stats['calls']) if api_stats['calls'] else 0.0
    _log_api(
        f"ALFRED API fetch DONE in {elapsed:.1f}s "
        f"calls={api_stats['calls']} ok={api_stats['ok']} err={api_stats['err']} avg_api={avg:.2f}s "
        f"final_min_interval={throttle['min_interval_s']:.2f}s"
    )
    if endpoint_counts:
        _log_api(f'Endpoint counts: {endpoint_counts}')
    if api_stats['err'] > 0:
        _log_api("API errors occurred. Inspect cache_payload.get('error_log', pd.DataFrame()).")

final_payload = hs.run_final_stage(
    registry=registry,
    cache_payload=cache_payload,
    final_path=FINAL_XLSX,
    start=start,
    end=end,
    max_release_gap_months=MAX_RELEASE_GAP_MONTHS,
 )

raw_bal = final_payload['snapshot_raw_balanced'].copy()
src_bal = final_payload['snapshot_source_balanced'].copy()
raw_bal['decision_date'] = pd.to_datetime(raw_bal['decision_date'])
src_bal['decision_date'] = pd.to_datetime(src_bal['decision_date'])
raw_bal = raw_bal.set_index('decision_date').sort_index()
src_bal = src_bal.set_index('decision_date').sort_index()

HIST_VINTAGE_DIRS = [
    ROOT / 'data/ALFRED/Historical-vintages-of-FRED-MD-1999-08-to-2024-12',
]
raw_bal, src_bal, filled_counts = hs.apply_anchor_backfill_to_balanced(
    raw_bal=raw_bal,
    src_bal=src_bal,
    series_names=series_names,
    nonrev_raw=nonrev_raw,
    vintage_dirs=HIST_VINTAGE_DIRS,
    tag='FILL_ANCHOR_VINTAGE_PRE_TSTOP',
)

rt_tcode = hs.apply_tcodes_panel(raw_bal, tcode_map, series_names)

raw_bal.to_csv(OUT_DIR / 'realtime_raw_balanced.csv', index_label='decision_date')
rt_tcode.to_csv(OUT_DIR / 'realtime_tcode_balanced.csv', index_label='decision_date')
src_bal.to_csv(OUT_DIR / 'realtime_source_tags.csv', index_label='decision_date')
cache_payload.get('availability_audit', pd.DataFrame()).to_csv(OUT_DIR / 'availability_audit.csv', index=False)
cache_payload.get('revision_audit', pd.DataFrame()).to_csv(OUT_DIR / 'revision_audit.csv', index=False)

source_counts = src_bal[series_names].stack().astype(str).value_counts()
source_tag_summary = pd.DataFrame({
    'source_tag': source_counts.index,
    'count': source_counts.values,
})
source_tag_summary['share_pct'] = (
    100.0 * source_tag_summary['count'] / source_tag_summary['count'].sum()
).round(2)

missing_by_series = raw_bal[series_names].isna().sum().astype(int)
dominant_source = {}
for s in series_names:
    vc = src_bal[s].astype(str).value_counts(dropna=False)
    dominant_source[s] = vc.index[0] if len(vc) else ''

series_quality = pd.DataFrame({
    'series': series_names,
    'missing_count': [int(missing_by_series[s]) for s in series_names],
    'missing_share_pct': [round(100.0 * missing_by_series[s] / len(raw_bal), 2) for s in series_names],
    'dominant_source_tag': [dominant_source[s] for s in series_names],
    'anchor_backfill_cells': [int(filled_counts.get(s, 0)) for s in series_names],
})

series_map = registry[[
    'mnemonic_hs',
    'tcode_hs',
    'construction_type',
    'target_series_id',
    'raw_formula',
    'revision_class',
]].rename(columns={
    'mnemonic_hs': 'series',
    'tcode_hs': 'tcode',
    'target_series_id': 'target_series',
})

overview = pd.DataFrame([
    {'metric': 'build_timestamp_utc', 'value': pd.Timestamp.now('UTC').isoformat()},
    {'metric': 'cache_mode', 'value': cache_mode},
    {'metric': 'decision_start', 'value': start},
    {'metric': 'decision_end', 'value': end},
    {'metric': 'series_count', 'value': int(len(series_names))},
    {'metric': 'decision_months', 'value': int(len(raw_bal))},
    {'metric': 'remaining_missing_cells', 'value': int(raw_bal[series_names].isna().sum().sum())},
    {'metric': 'anchor_backfill_series_count', 'value': int(len(filled_counts))},
    {'metric': 'anchor_backfill_cell_count', 'value': int(sum(filled_counts.values()))},
    {'metric': 'api_calls', 'value': int(api_stats['calls'])},
    {'metric': 'api_errors', 'value': int(api_stats['err'])},
])

readme = pd.DataFrame([
    {'section': 'Use this workbook for', 'detail': 'Quick understanding of data quality and final transformed panel.'},
    {'section': 'Main data sheet', 'detail': 'rt_tcode_balanced (final transformed monthly panel used for modeling).'},
    {'section': 'Data source mix', 'detail': 'source_tag_summary shows which fill/source tags dominate the panel.'},
    {'section': 'Series-level health', 'detail': 'series_quality highlights missingness and dominant source per series.'},
    {'section': 'Raw/source full matrices', 'detail': 'See CSV outputs in data/ALFRED/simple_outputs for full raw and source-tag matrices.'},
])

diag_checks = final_payload.get('diag_checks', pd.DataFrame()).copy()
error_log = cache_payload.get('error_log', pd.DataFrame()).copy()
if error_log.empty:
    error_log_summary = pd.DataFrame([{'status': 'no_errors_logged'}])
else:
    keep_cols = [c for c in ['stage', 'series_id', 'error'] if c in error_log.columns]
    error_log_summary = error_log[keep_cols].copy()
    if 'error' in error_log_summary.columns:
        error_log_summary['error'] = error_log_summary['error'].astype(str).str.slice(0, 200)

raw_last24 = raw_bal.tail(24).reset_index()
source_last24 = src_bal.tail(24).reset_index()

sheet_plan = [
    'readme',
    'overview',
    'series_map',
    'series_quality',
    'source_tag_summary',
    'rt_tcode_balanced',
    'rt_raw_last24m',
    'rt_source_last24m',
    'diag_checks',
    'error_log',
]

with pd.ExcelWriter(FINAL_XLSX, engine='openpyxl') as w:
    readme.to_excel(w, sheet_name='readme', index=False)
    overview.to_excel(w, sheet_name='overview', index=False)
    series_map.to_excel(w, sheet_name='series_map', index=False)
    series_quality.to_excel(w, sheet_name='series_quality', index=False)
    source_tag_summary.to_excel(w, sheet_name='source_tag_summary', index=False)
    rt_tcode.reset_index().to_excel(w, sheet_name='rt_tcode_balanced', index=False)
    raw_last24.to_excel(w, sheet_name='rt_raw_last24m', index=False)
    source_last24.to_excel(w, sheet_name='rt_source_last24m', index=False)
    diag_checks.to_excel(w, sheet_name='diag_checks', index=False)
    error_log_summary.to_excel(w, sheet_name='error_log', index=False)

print({
    'cache_mode': cache_mode,
    'final_xlsx': str(FINAL_XLSX),
    'csv_dir': str(OUT_DIR),
    'xlsx_sheets': sheet_plan,
    'remaining_missing_cells': int(raw_bal[series_names].isna().sum().sum()),
})

[2026-03-20 10:22:46] Starting ALFRED API fetch stage (run_cache_stage). base_min_interval=0.55s cooldown_on_429=12s
[2026-03-20 10:22:47] API OK #1 endpoint=series/observations series=AAA dt=0.57s ok=1 err=0 min_interval=0.55s
[2026-03-20 10:23:10] API OK #25 endpoint=series/observations series=CPIMEDSL dt=0.37s ok=25 err=0 min_interval=0.55s
[2026-03-20 10:23:36] API OK #50 endpoint=series/observations series=HOUST dt=0.38s ok=50 err=0 min_interval=0.55s
[2026-03-20 10:24:00] API OK #75 endpoint=series/observations series=MANEMP dt=0.37s ok=75 err=0 min_interval=0.55s
[2026-03-20 10:24:38] API ERROR #92 endpoint=series/observations series=SP500 dt=17.72s error=FredApiError: Request failed after 6 attempts: series/observations
[2026-03-20 10:24:48] API OK #100 endpoint=series/observations series=TB6MS dt=0.38s ok=99 err=1 min_interval=0.55s
[2026-03-20 10:25:25] API ERROR #120 endpoint=series/observations series=VIXCLS dt=16.99s error=FredApiError: Request failed after 6 attempts: ser

KeyboardInterrupt: 

## Step 4: Quick Sanity Summary

This cell prints a compact quality summary: number of series, row counts, remaining missing values, and the top source tags in the balanced real-time panel.

Use it as a lightweight check before consuming the final workbook in downstream modeling.

In [10]:
summary = pd.DataFrame([
    {'metric': 'series_total', 'value': len(series_names)},
    {'metric': 'nonrevised_rows', 'value': int(len(nonrev_raw))},
    {'metric': 'realtime_rows', 'value': int(len(raw_bal))},
    {'metric': 'realtime_missing_after_balance', 'value': int(raw_bal[series_names].isna().sum().sum())},
])
print(summary.to_string(index=False))
print('Top source tags:')
print(src_bal[series_names].stack().astype(str).value_counts().head(12).to_string())

                        metric  value
                  series_total    126
               nonrevised_rows    805
                 realtime_rows    805
realtime_missing_after_balance   1462
Top source tags:
FILL_PRE_VINTAGE_EARLIEST        48654
STRICT                           44606
FILL_ANCHOR_VINTAGE_PRE_TSTOP     6708
missing_after_fill                1462
